# Search-R1 실제 데이터 학습: NQ + HotpotQA
GPU 서버에서 공식 QA Parquet 준비 → Wikipedia 디스크 검색 인덱스 → 형식 SFT → PPO → 평가를 실행합니다.
모델은 HyperCLOVA 1.5B + LoRA, 검색은 SQLite FTS5 BM25입니다. 원 논문의 E5 검색/모델과 다른 실험입니다.

먼저 README의 설치 절차를 완료하고 **서버의 Python 3.10/3.11 GPU 커널**을 선택하세요.
기존 데모 노트북은 `train_demo.ipynb`, 기존 체크포인트는 기존 outputs 폴더에 보존됩니다.
수정 파일을 서버에 복사하고 커널을 재시작해야 반영됩니다.

## 1. 커널과 환경 확인
이 노트북과 `policy_r1` 폴더가 같은 디렉터리에 있어야 합니다. Python 3.10 또는 3.11을 사용합니다.

In [ ]:
# 역할: 현재 커널의 Python과 프로젝트 위치를 확인합니다.
import sys
from pathlib import Path
print("Python:", sys.version)
print("Kernel executable:", sys.executable)
assert (3, 10) <= sys.version_info[:2] <= (3, 11), "Python 3.10/3.11 커널을 선택하세요."
PROJECT_ROOT = Path.cwd()
assert (PROJECT_ROOT / "policy_r1").is_dir(), "노트북을 Search-R1 프로젝트 루트에서 실행하세요."
print("Project:", PROJECT_ROOT)

In [ ]:
# 데이터 준비 전에는 GPU/패키지만 확인합니다.
import subprocess
subprocess.run([sys.executable, "scripts/check_environment.py", "--hardware-only"], check=True)

설치가 안 되어 있다면 README의 서버 터미널 명령으로 설치하세요. 설치/버전 변경 후에는 커널을 재시작합니다. `CUDA GPU 없음`이면 서버의 커널과 NVIDIA 드라이버를 확인합니다.

## 2. 모델 접근 인증
[모델 페이지](https://huggingface.co/naver-hyperclovax/HyperCLOVAX-SEED-Text-Instruct-1.5B)에서 이용 조건에 동의한 계정의 Hugging Face read token을 사용하세요. 토큰은 숨김 입력으로 받으며 셀에 직접 적지 않습니다.

In [ ]:
# 역할: Hugging Face 인증이 없으면 토큰을 숨김 입력으로 받아 로그인합니다.
from getpass import getpass
from huggingface_hub import get_token, login
if not get_token():
    login(token=getpass("Hugging Face read token: "), add_to_git_credential=False)
print("인증 설정 완료. 모델 접근 권한은 다음 로딩 단계에서 확인합니다.")

## 3. 본 학습 설정
데모 데이터는 허용하지 않습니다. 아래 `RESUME_FROM`은 새 학습이면 None입니다.
재개할 때는 본 학습 체크포인트 경로를 지정하고 `DATA_PASSES`를 이전 목표보다 늘리세요.
데이터 1회 순회는 약 2만 PPO update이며, 단일 GPU에서는 장시간 실행됩니다.
`ppo_epochs`는 수집한 배치의 재사용 횟수로 `DATA_PASSES`와 다릅니다.

In [ ]:
from datetime import datetime
import json
import math
from policy_r1.config import Config, load_config
from policy_r1.data import load_qa

RESUME_FROM = None  # 예: "outputs/search_r1_날짜_시간/checkpoint-00500"
DATA_PASSES = 1  # 재개 시 완료분을 포함한 총 목표 순회 수
if RESUME_FROM:
    config = Config(**json.loads((Path(RESUME_FROM) / "config.json").read_text()))
else:
    config = load_config("configs/search_r1.yaml")
assert not config.allow_demo and config.train_sampling == "shuffle", "본 학습 설정을 사용하세요."
config.output_dir = f"outputs/search_r1_{datetime.now():%Y%m%d_%H%M%S_%f}"
config.validate()
print("실행 설정:", config.to_dict())

### 3.1. 데이터 다운로드·변환·검색 인덱스 준비
아래 셀은 없는 원본 파일을 다운로드하고, QA를 JSONL로 변환한 뒤 Wikipedia 압축 파일을 스트리밍하여 SQLite 검색 인덱스를 만듭니다.
QA 원본은 train 169,615행/test 51,713행입니다. 현재 파일 기준 중복·겹침 정리 후 train 167,391 / val 2,000 / test 50,480입니다.
원본 test는 튜닝에 사용하지 않습니다. 원본 Parquet는 그대로 두며 변환본 test는 중복 질문을 합친 별도 평가셋입니다.

Wikipedia 압축 파일은 약 5.12 GB이며 인덱스는 추가 공간과 생성 시간이 필요합니다. 서버 디스크 여유를 먼저 확인하세요.
동일 입력의 완료된 변환·인덱스는 재사용합니다. 중단된 임시 인덱스는 학습용으로 사용하지 않습니다.
별도 터미널에서 `python scripts/prepare_search_r1.py --download`를 먼저 실행해도 됩니다.


In [ ]:
# 최초 실행 시 전체 데이터 준비. 이미 완료됐다면 검증 후 재사용합니다.
# 경로를 바꿨다면 아래 준비 CLI 옵션과 config의 경로를 함께 맞추세요.
PREPARE_DATA = True
if PREPARE_DATA:
    subprocess.run([sys.executable, "scripts/prepare_search_r1.py", "--stage", "all", "--download"], check=True)


In [ ]:
# 준비된 파일과 검색 인덱스를 검사합니다. 인덱스는 Python 메모리로 전체 로드하지 않습니다.
from policy_r1.data import load_datasets
from policy_r1.retrieval import build_retriever
corpus, train_rows, val_rows = load_datasets(config)
retriever = build_retriever(config, corpus)
print(f"검색 문서 {len(corpus):,}, 학습 {len(train_rows):,}, 검증 {len(val_rows):,}")
print(f"최종 test {len(load_qa(config.test_path)):,} (학습에 사용하지 않음)")
assert DATA_PASSES >= 1
config.total_updates = math.ceil(len(train_rows) * DATA_PASSES / config.rollout_batch_size)
print("PPO 총 목표:", config.total_updates, "updates / 배치:", config.rollout_batch_size)
print("주기 평가:", min(config.eval_max_samples or len(val_rows), len(val_rows)), "문항의 고정 subset")
hits = retriever.search(train_rows[0]["question"], config.top_k)
print("검색 질문:", train_rows[0]["question"])
for hit in hits:
    print(hit["id"], round(hit["score"], 3), hit["contents"])
assert hits, "검색 결과가 없습니다. 인덱스와 질문을 점검하세요."
if hasattr(retriever, "close"):
    retriever.close()
del retriever, corpus

## 4. Actor와 독립 Critic 로딩
두 HyperCLOVA backbone을 별도로 로드하고 각각 LoRA를 붙입니다. Critic에는 Value Head를 추가합니다. Reference는 Actor의 고정 base를 공유하며 adapter를 끈 상태로 계산합니다. 첫 다운로드에는 시간이 걸릴 수 있습니다.

In [ ]:
from policy_r1.trainer import PPOExperiment
experiment = PPOExperiment(config, resume_from=RESUME_FROM)
assert experiment.step < config.total_updates, "목표 순회 수 DATA_PASSES를 늘리세요."
print("절대 저장 경로:", experiment.output.resolve())

## 5. 형식 학습 전 생성과 평가
원하는 동작은 `<search>` → 실제 `<information>` 관측 → `<answer>`입니다.
`assistant -> tool/python`, `search(...)`, 코드블록과 모델이 쓴 가짜 검색 결과는 검색으로 실행하지 않습니다.
`hit_token_limit=True`와 `generation_limit`는 생성 길이 제한에 도달한 경우입니다. 태그가 잘못된 채 192토큰을 다 쓴다면, 길이만 늘리기 전에 형식을 교정해야 합니다.


In [ ]:
# 역할: 학습 전 생성과 검색 trace를 확인합니다.
trajectory = experiment.rollout(val_rows[0]["question"], sample=False)
for event in trajectory.events:
    print(event)
print("종료 사유:", trajectory.finish_reason, "최종 답변:", trajectory.answer)
print("전체 토큰:", len(trajectory.input_ids), "PPO action 토큰:", sum(trajectory.action_mask))

In [ ]:
# 새 실험: 초기 base / 재개: 저장된 Actor. 동일한 고정 validation subset으로 비교합니다.
base_before = experiment.evaluate()
print("이번 실행 시작 시 평가:", base_before)

## 5.1. 실제 검색에 기반한 형식 SFT
학습 split에서 seed로 최대 2,048개 후보를 뽑고, 잘린 실제 검색 관측에 정답이 보이는 예제를 최대 128개 사용합니다.
공식 QA에는 Wikipedia evidence_ids가 없으므로 이를 강제하지 않지만, 검색 관측에 정답이 없으면 SFT 예제에서 제외합니다.
이는 문자열에 기반한 형식 학습용 필터이며 근거의 의미적 충분성을 보장하지는 않습니다.
검증/test 정답으로 SFT 예제를 만들거나 정답을 PPO 응답에 넣지 않습니다. PPO는 전체 학습 질문을 사용합니다.

기본 SFT는 128 update이며 재개 시 건너뜁니다. 유효한 답변이 나오지 않으면 다음 점검 셀이 중단합니다.
검색 trace와 형식/길이 설정을 확인하세요. 초기 base → SFT → PPO 평가는 별도로 기록합니다.

In [ ]:
RUN_PROTOCOL_WARMUP = True
WARMUP_STEPS = 128
if RUN_PROTOCOL_WARMUP and RESUME_FROM is None:
    warmup_history = experiment.warmup_protocol(steps=WARMUP_STEPS, learning_rate=1e-4)
    if not (experiment.output / "checkpoint-00000").exists():
        experiment.save()  # PPO 전 SFT 결과도 보존
print("누적 형식 SFT update:", experiment.protocol_warmup_steps)

In [ ]:
# 역할: 실제 sampled rollout을 점검하고 학습 신호 없이 PPO를 시작하지 않도록 합니다.
report = experiment.diagnose_rollouts(count=8, sample=True)
print(report["summary"])
for example in report["examples"]:
    print(example["id"], example["finish_reason"], "reward=", example["reward"])
    for event in example["events"]:
        print(event)
if report["summary"]["valid_answer"] == 0:
    raise RuntimeError("8개 sampled rollout에 유효한 answer가 없습니다. trace의 형식/길이를 먼저 수정하거나, PPO 시작 전 형식 warmup을 보완하세요. 아직 학습 성공이 아닙니다.")
# 이 값과 학습 후 지표를 비교해야 PPO만의 변화를 구분할 수 있습니다.
before = experiment.evaluate()
print("형식 학습 후 / PPO 전:", before)


## 6. PPO 본 학습
train split을 섞어서 중복 없이 순회합니다. 마지막 배치는 다음 순회의 일부가 포함될 수 있습니다.
`train_examples_seen`, `train_passes`로 실제 사용량을 확인하세요. 10 update마다 저장하고, 500 update마다 고정 검증 subset을 평가합니다.
마지막 update에도 저장합니다. `eval_count`와 학습 배치 EM을 구분하세요.

Actor/Critic gradient가 모두 0인 상태가 연속되면 예외로 중단합니다. 데이터가 많아도 학습 신호가 없으면
성공이 아닙니다. `diagnostics.jsonl`, `rollouts.jsonl`, `warmup.jsonl`을 확인하세요.

In [ ]:
# 역할: rollout → reference KL/GAE → clipped PPO·Value loss 업데이트를 반복합니다.
history = experiment.train()

## 7. 학습 후 평가와 학습 곡선

In [ ]:
after = experiment.evaluate()
print("이번 실행 시작 시 / 동일 subset:", base_before)
print("PPO 전 / 동일 subset:", before)
print("PPO 후 / 동일 subset:", after)
print("전체 검증:", experiment.evaluate(full=True))
print("저장 경로:", experiment.output.resolve())
checkpoint = experiment.output / f"checkpoint-{experiment.step:05d}"
print("Actor 저장 확인:", (checkpoint / "actor" / "adapter_model.safetensors").is_file())
# test는 최종 모델/설정을 확정한 뒤 한 번 실행합니다.
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    print("최종 test:", experiment.evaluate(split="test"))

In [ ]:
# 역할: loss와 실제 gradient, 정답률을 함께 확인합니다.
import json
import matplotlib.pyplot as plt
rows = [json.loads(line) for line in (Path(config.output_dir) / "metrics.jsonl").read_text().splitlines()]
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for ax, key in zip(axes.flat, ["actor_loss", "critic_loss", "exact_match", "actor_grad_norm", "critic_grad_norm", "valid_answer"]):
    ax.plot([r["step"] for r in rows], [r[key] for r in rows], marker="o")
    ax.set(title=key, xlabel="PPO update")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# 역할: 학습된 Actor로 원하는 정책 질문에 답하고 검색 근거 trace를 확인합니다.
question = val_rows[0]["question"]
result = experiment.rollout(question, sample=False)
print("답변:", result.answer)
for event in result.events:
    print(event)

## 8. 저장된 본 학습 재개와 최종 평가
새 커널에서 시작하고 3번 설정 셀의 `RESUME_FROM`에 체크포인트 경로를 넣으세요.
`DATA_PASSES`는 완료한 학습을 포함한 총 목표입니다. 이전 목표보다 늘려서 실행하세요.
체크포인트에는 LoRA, Critic value head, optimizer, RNG, 남은 질문 순서와 누적 사용량이 저장됩니다.
데모 체크포인트는 데이터·언어·길이 설정이 달라 이 본 학습에 그대로 재개하지 않습니다.

터미널에서도 같은 작업을 실행할 수 있습니다.
```bash
python scripts/train.py --resume-from outputs/실험명/checkpoint-00500 --epochs 2
python scripts/evaluate.py --checkpoint outputs/실험명/checkpoint-20924 --split test
```
평가 요약은 `evaluation.jsonl`(데이터셋별 점수 포함), 개별 응답은 `evaluation_examples.jsonl`입니다.
최종 체크포인트 번호는 데이터 개수·배치 크기·목표 순회 수에 따라 달라집니다.
로컬 CPU 테스트와 실제 Parquet 변환은 검증할 수 있지만 전체 Wikipedia 인덱스·CUDA 학습은 서버에서 실행해야 합니다.